In [17]:
# ─── Imports ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120

In [18]:
# ─── Load Raw Data ─────────────────────────────────────────
# We always work from raw data in the cleaning notebook
# Never overwrite raw data — always save cleaned version separately
# This is a professional standard: raw data is sacred

df_1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df_2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df = pd.concat([df_1, df_2], ignore_index=True)

print(f"Raw data shape: {df.shape}")
print(f"Raw rows: {df.shape[0]:,}")

Raw data shape: (1067371, 8)
Raw rows: 1,067,371


In [19]:
# ─── Snapshot Before Cleaning ──────────────────────────────
# We document the before state professionally
# This lets us calculate exactly how much data we removed and why
# A cleaning log builds trust with stakeholders

snapshot = {
    'Total rows': len(df),
    'Unique customers': df['Customer ID'].nunique(),
    'Unique invoices': df['Invoice'].nunique(),
    'Missing Customer ID': df['Customer ID'].isnull().sum(),
    'Duplicates': df.duplicated().sum(),
    'Cancellations': df['Invoice'].astype(str).str.startswith('C').sum(),
    'Negative quantity': (df['Quantity'] < 0).sum(),
    'Zero price': (df['Price'] == 0).sum(),
    'Negative price': (df['Price'] < 0).sum(),
}

print("BEFORE CLEANING")
print("=" * 45)
for k, v in snapshot.items():
    print(f"{k:<25} {v:>10,}")

BEFORE CLEANING
Total rows                 1,067,371
Unique customers               5,942
Unique invoices               53,628
Missing Customer ID          243,007
Duplicates                    34,335
Cancellations                 19,494
Negative quantity             22,950
Zero price                     6,202
Negative price                     5


In [20]:
# ─── Step 1: Drop Missing Customer IDs ─────────────────────
# WHY: Customer ID is the backbone of RFM analysis
# Without it we cannot assign any transaction to a customer
# 22.77% loss is significant but unavoidable — documented in EDA

before = len(df)
df = df.dropna(subset=['Customer ID'])
after = len(df)

print(f"Rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")
print(f"% removed: {(before - after) / before * 100:.2f}%")

Rows removed: 243,007
Rows remaining: 824,364
% removed: 22.77%


In [21]:
# ─── Step 2: Drop Duplicates ───────────────────────────────
# WHY: Duplicate rows inflate transaction counts
# A customer appearing twice for the same purchase = 
# double counted frequency and revenue in RFM
# We keep the first occurrence

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Duplicates removed: {before - after:,}")
print(f"Rows remaining: {after:,}")
print(f"% Duplicate percentage: {(before - after) / before*100:.2f}%") 

Duplicates removed: 26,479
Rows remaining: 797,885
% Duplicate percentage: 3.21%


In [22]:
# ─── Step 3: Remove Cancellations ──────────────────────────
# WHY: Invoices starting with 'C' are returns/cancellations
# These represent negative revenue events
# Including them would undercount frequency and distort monetary value
# RFM should only reflect actual purchase behaviour

before = len(df)
df = df[~df['Invoice'].astype(str).str.startswith('C')]
after = len(df)

print(f"Cancellations removed: {before - after:,}")
print(f"Rows remaining: {after:,}")
print(f"% C percentage: {(before - after)/before * 100:.2f}%")

Cancellations removed: 18,390
Rows remaining: 779,495
% C percentage: 2.30%


In [23]:
# ─── Step 4: Remove Invalid Quantities & Prices ────────────
# WHY: Negative quantities outside cancellations = manual adjustments
# Zero/negative prices = free samples, internal transfers, accounting entries
# None of these represent real customer purchase behaviour

before = len(df)
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
after = len(df)

print(f"Invalid rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")

Invalid rows removed: 70
Rows remaining: 779,425


In [24]:
# ─── Step 5: Fix Data Types ────────────────────────────────
# WHY: Customer ID was float64 (e.g. 12345.0) — not meaningful as a number
# Converting to string/int removes the decimal and treats it as an identifier
# Invoice should also be string for consistent filtering
# Correct dtypes prevent silent bugs downstream

df['Customer ID'] = df['Customer ID'].astype(int).astype(str)
df['Invoice'] = df['Invoice'].astype(str)

# Add revenue column — needed for Monetary in RFM
df['Revenue'] = df['Quantity'] * df['Price']

print("Updated dtypes:")
print(df.dtypes)
print(f"\nSample Customer IDs: {df['Customer ID'].head().tolist()}")

Updated dtypes:
Invoice                   str
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
Revenue               float64
dtype: object

Sample Customer IDs: ['13085', '13085', '13085', '13085', '13085']


In [25]:
# ─── Step 6: Outlier Detection & Treatment ─────────────────
# WHY: Extreme outliers in Quantity and Price can be:
# - Bulk B2B orders (legitimate but distort RFM scores)
# - Data entry errors (e.g. 80,000 units of one item)
# We use IQR method to detect and cap (not remove) outliers
# Capping preserves the row but limits extreme influence

def cap_outliers(series, lower_q=0.01, upper_q=0.99):
    """Cap outliers at 1st and 99th percentile (Winsorization)"""
    lower = series.quantile(lower_q)
    upper = series.quantile(upper_q)
    return series.clip(lower=lower, upper=upper)

# Show outlier boundaries before capping
for col in ['Quantity', 'Price', 'Revenue']:
    q1 = df[col].quantile(0.01)
    q99 = df[col].quantile(0.99)
    outliers = df[(df[col] < q1) | (df[col] > q99)].shape[0]
    print(f"{col}: 1st pct={q1:.2f}, 99th pct={q99:.2f}, outliers={outliers:,}")

# Cap outliers
df['Quantity'] = cap_outliers(df['Quantity'])
df['Price'] = cap_outliers(df['Price'])
df['Revenue'] = df['Quantity'] * df['Price']  # recalculate after capping

print("\nOutliers capped using Winsorization (1st-99th percentile)")

Quantity: 1st pct=1.00, 99th pct=144.00, outliers=5,876
Price: 1st pct=0.29, 99th pct=14.95, outliers=14,455
Revenue: 1st pct=0.60, 99th pct=203.52, outliers=15,542

Outliers capped using Winsorization (1st-99th percentile)


In [31]:
# ─── Snapshot After Cleaning ───────────────────────────────
# We document the after state and calculate total data loss
# This is what you present to stakeholders:
# "We removed X rows for these specific reasons"

snapshot_after = {
    'Total rows': len(df),
    'Unique customers': df['Customer ID'].nunique(),
    'Unique invoices': df['Invoice'].nunique(),
    'Missing Customer ID': df['Customer ID'].isnull().sum(),
    'Duplicates': df.duplicated().sum(),
    'Negative quantity': (df['Quantity'] < 0).sum(),
    'Zero/negative price': (df['Price'] <= 0).sum(),
}


print("AFTER CLEANING")
print("=" * 45)
for k, v in snapshot_after.items():
    print(f"{k:<25} {v:>10,}")

total_removed = 1067371 - len(df)
print(f"\nTotal rows removed: {total_removed:,}")
print(f"Data retained: {len(df)/1067371*100:.1f}%")

AFTER CLEANING
Total rows                   779,425
Unique customers               5,878
Unique invoices               36,969
Missing Customer ID                0
Duplicates                        15
Negative quantity                  0
Zero/negative price                0

Total rows removed: 287,946
Data retained: 73.0%


##### ─── Cleaning Summary Visualization ────────────────────────
# A visual summary of what we removed and why
# This goes into the README and makes the project look professional

labels = ['Missing\nCustomer ID', 'Duplicates', 'Cancellations', 
          'Invalid Qty\n& Price', 'Clean Data']
values = [243007, 34335, 19494, 9659, len(df)]
colors = ['#E74C3C', '#E67E22', '#F39C12', '#E74C3C', '#27AE60']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Data Cleaning Summary — Rows Removed by Reason', fontsize=14, pad=15)
ax.set_ylabel('Number of Rows')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../reports/cleaning_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved to reports/")

In [34]:
# ─── Save Clean Data ───────────────────────────────────────
# Save to data/processed/ — never overwrite raw data
# CSV is universally readable and fast to load in future notebooks
# We also save a summary of the clean dataset

df.to_csv('C:/Users/DELL/rfm-analysis/data/processed/clean_retail.csv', index=False)

print("Clean data saved to data/processed/clean_retail.csv")
print(f"\nFinal dataset shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nSample data:")
df.head()

Clean data saved to data/processed/clean_retail.csv

Final dataset shape: (779425, 9)

Column dtypes:
Invoice                   str
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID               str
Country                   str
Revenue               float64
dtype: object

Sample data:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00


In [35]:
# ─── Write src/cleaning.py ─────────────────────────────────
# We now package our cleaning logic into a reusable module
# This is what separates a notebook project from an engineering project

cleaning_code = '''import pandas as pd
import numpy as np


def load_raw_data(path: str) -> pd.DataFrame:
    """Load and combine both sheets from the UCI Online Retail II dataset."""
    df_1 = pd.read_excel(path, sheet_name="Year 2009-2010")
    df_2 = pd.read_excel(path, sheet_name="Year 2010-2011")
    return pd.concat([df_1, df_2], ignore_index=True)


def cap_outliers(series: pd.Series, lower_q=0.01, upper_q=0.99) -> pd.Series:
    """Winsorize a series at given quantile boundaries."""
    return series.clip(lower=series.quantile(lower_q),
                       upper=series.quantile(upper_q))


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full cleaning pipeline for UCI Online Retail II dataset.
    Steps:
        1. Drop missing Customer IDs
        2. Drop duplicates
        3. Remove cancellations (Invoice starts with C)
        4. Remove invalid quantities and prices
        5. Fix data types
        6. Add Revenue column
        7. Cap outliers via Winsorization
    Returns cleaned DataFrame.
    """
    df = df.copy()

    # Step 1: Drop missing Customer IDs
    df = df.dropna(subset=["Customer ID"])

    # Step 2: Drop duplicates
    df = df.drop_duplicates()

    # Step 3: Remove cancellations
    df = df[~df["Invoice"].astype(str).str.startswith("C")]

    # Step 4: Remove invalid quantities and prices
    df = df[df["Quantity"] > 0]
    df = df[df["Price"] > 0]

    # Step 5: Fix data types
    df["Customer ID"] = df["Customer ID"].astype(int).astype(str)
    df["Invoice"] = df["Invoice"].astype(str)

    # Step 6: Add Revenue column
    df["Revenue"] = df["Quantity"] * df["Price"]

    # Step 7: Cap outliers
    df["Quantity"] = cap_outliers(df["Quantity"])
    df["Price"] = cap_outliers(df["Price"])
    df["Revenue"] = df["Quantity"] * df["Price"]

    return df


if __name__ == "__main__":
    RAW_PATH = "../data/raw/online_retail_II.xlsx"
    OUTPUT_PATH = "../data/processed/clean_retail.csv"

    print("Loading raw data...")
    df_raw = load_raw_data(RAW_PATH)
    print(f"Raw shape: {df_raw.shape}")

    print("Cleaning data...")
    df_clean = clean_data(df_raw)
    print(f"Clean shape: {df_clean.shape}")

    df_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved to {OUTPUT_PATH}")
'''

with open('../src/cleaning.py', 'w') as f:
    f.write(cleaning_code)

print("src/cleaning.py created successfully!")

src/cleaning.py created successfully!
